In [6]:
# Imports
from gliner import GLiNER

import os
import sys
import dotenv

import json
from pathlib import Path

from collections import defaultdict
from prettytable import PrettyTable

dotenv.load_dotenv()
ROOT_DIR = os.environ.get("ROOT_DIR")
sys.path.append(f"{ROOT_DIR}/scripts")

from evaluation import evaluate

In [7]:
test_before_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_before_2000.json", "r"))
test_after_2000 = json.load(open(f"{ROOT_DIR}/data/splits/test_after_2000.json", "r"))

# Raw model

## Inference 

In [8]:
def inference(model, data, tags, threshold):
    all_doc = []
    for doc in data:
        text = doc["texte"]
        entities = []
        # Diviser le texte en chunks si trop long
        max_length = 300  # caractères (pas tokens)
        chunks = []
        start_pos = 0
        while start_pos < len(text):
            # Trouver un point ou une virgule pour couper proprement
            end_pos = min(start_pos + max_length, len(text))
            if end_pos < len(text):
                # Chercher le dernier point/virgule/espace avant la limite
                last_punct = max(
                    text.rfind('. ', start_pos, end_pos),
                    text.rfind('! ', start_pos, end_pos),
                    text.rfind('? ', start_pos, end_pos),
                    text.rfind('\n', start_pos, end_pos)
                )
                if last_punct > start_pos:
                    end_pos = last_punct + 1
            chunk = text[start_pos:end_pos].strip()
            if chunk:
                chunks.append((chunk, start_pos))
            start_pos = end_pos
        
        # Prédire sur chaque chunk
        for chunk_text, chunk_offset in chunks:
            try:
                detected_entities = model.predict_entities(
                    text=chunk_text, 
                    labels=tags, 
                    threshold=threshold
                )
                for ent in detected_entities:
                    # Mapper les labels GLiNER aux tags NER standard
                    label_mapping = {
                        "person": "PER",
                        "location": "LOC",
                        "political party/political movement": "ORG",
                        "profession": "MISC"
                    }
                    
                    original_label = ent["label"].lower()
                    mapped_tag = label_mapping.get(original_label, "MISC")
                    
                    entities.append({
                        "texte": ent["text"],
                        "tag": mapped_tag,
                        "debut": ent["start"] + chunk_offset,
                        "fin": ent["end"] + chunk_offset
                    })
            except Exception as e:
                print(f"Erreur sur chunk {doc['id']}: {e}")
        
        all_doc.append({
            "id": doc["id"],
            "annee": doc["annee"],
            "predicted_entities": entities
        })
    return all_doc

In [9]:
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1") #urchade/gliner_base # urchade/gliner_multi_pii-v1
tags = ["person", "political party/political movement", "location", "profession"]

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]


/Users/wiamlachqer/Desktop/Named_Entity_Extraction_Archelec_Corpus/projet_archelec/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [11]:
for threshold in thresholds:
    print(f"----------------- Running inference with threshold {threshold} -----------------")

    prediction_before_2000 = inference(model, test_before_2000, tags=tags, threshold=threshold)
    prediction_after_2000 = inference(model, test_after_2000, tags=tags, threshold=threshold)

    metrics_before_2000 = evaluate(prediction_before_2000, test_before_2000)
    metrics_after_2000 = evaluate(prediction_after_2000, test_after_2000)

    # Chemin du fichier
    output_path_before_2000 = "../data/results/Gliner/raw/metrics_before_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_before_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_before_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_before_2000, f, indent=2, ensure_ascii=False)

    # Chemin du fichier
    output_path_after_2000 = "../data/results/Gliner/raw/metrics_after_2000_{}.json".format(threshold)

    # Créer les répertoires s'ils n'existent pas
    os.makedirs(os.path.dirname(output_path_after_2000), exist_ok=True)

    # Sauvegarder le JSON
    with open(output_path_after_2000, 'w', encoding='utf-8') as f:
        json.dump(metrics_after_2000, f, indent=2, ensure_ascii=False)

----------------- Running inference with threshold 0.1 -----------------
Global NER Performance (Exact Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.0835 |
|   Recall  | 0.3855 |
|  F1-Score | 0.1373 |
+-----------+--------+

Global NER Performance (Partial Match)
+-----------+--------+
|   Metric  | Value  |
+-----------+--------+
| Precision | 0.1478 |
|   Recall  | 0.6822 |
|  F1-Score | 0.2430 |
+-----------+--------+

Performance by Tag — Exact Match
+------+-----------+--------+----------+---------+------------+
| Tag  | Precision | Recall | F1-Score | Support | Partial TP |
+------+-----------+--------+----------+---------+------------+
| LOC  |   0.0519  | 0.4321 |  0.0926  |    81   |     47     |
| MISC |   0.0767  | 0.3415 |  0.1253  |    82   |     17     |
| ORG  |   0.1070  | 0.2194 |  0.1438  |   196   |     37     |
| PER  |   0.1107  | 0.8551 |  0.1960  |    69   |     26     |
+------+-----------+--------+----------+------

# Fine tuned model

## Training